In [1]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124
12.4


In [2]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [3]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [5]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)


2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [6]:
from model_PyG import *
from utils import *

In [7]:
import copy
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T
import os

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam, AdamW

In [8]:
import torch_geometric
print(torch_geometric.__version__)

""" np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False """

2.7.0


' np.random.seed(0)\ntorch.manual_seed(0)\ntorch.cuda.manual_seed(0)\ntorch.cuda.manual_seed_all(0)\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.benchmark = False '

### Parameters

In [9]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

epochs = params["epochs"]
print("Epochs:\t", epochs)

cuda = params["cuda"]
print("Cuda:\t\t", cuda)

epochs = params["epochs"]
print("Epochs:\t\t", epochs)

lr = params["lr"]
print("Lr:\t\t", lr)

weight_decay = params["weight_decay"]
print("Weight decay:\t", weight_decay)

patience = params["patience"]
print("Patience:\t", patience)

Exp:		 exp106
Raw data:	 ACM-DBLP
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 128
Groups id:	 ['ACM-DBLP']
Subgroups id:	 {'ACM-DBLP': ['1', '2']}
Epochs:	 500
Cuda:		 1
Epochs:		 500
Lr:		 0.0001
Weight decay:	 0.0005
Patience:	 10


### Setup

In [10]:
# Parameters models

dataset = exp
encoders = ["GCNRes", "GIN", "GINE", "GCN", "GAT", "GATE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

KS = (1, 5, 10, 50)

In [11]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['ACM-DBLP_1', 'ACM-DBLP_2']]

### Create Data (PyG)

In [12]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	# T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [13]:
print("Loading training datasets")
has_edge_attributes = False
with_mean1 = True
with_mean2 = True

list_train_loader = []

if raw_data_file == "ACM_DBLP":
	group_id = groups_id[0]
	
	# Read dataset
	b = np.load("data/ACM-DBLP.npz")

	# Node matching
	test_pairs = b["test_pairs"].astype(np.int32)
	pos_pairs = b["pos_pairs"].astype(np.int32)
	df_node_alignment_truth = pd.DataFrame(np.concatenate((test_pairs, pos_pairs), axis=0))
	df_node_alignment_truth

	# Data preparation
	subgroup_id = 1
	edge_index1 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x1 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

	subgroup_id = 2
	edge_index2 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x2 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

elif raw_data_file == "Douban Online_Offline":
	group_id = groups_id[0]
	
	# Read dataset
	a1, f1, a2, f2, test_pairs = load_douban()

	# Node matching
	test_pairs = torch.tensor(np.array(test_pairs, dtype=int)) - 1
	test_pairs = test_pairs.numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs.T)
	df_node_alignment_truth

	# Create data
	edge_index1, _ = dense_to_sparse(torch.from_numpy(a1.toarray()))
	x1 = torch.from_numpy(f1.toarray()).float()

	edge_index2, _= dense_to_sparse(torch.from_numpy(a2.toarray()))
	x2 = torch.from_numpy(f2.toarray()).float()

	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "Cora1-Cora2":
	group_id = groups_id[0]
		
	# Read dataset
	loader = load_npz("data/cora.npz")
	data = loader["adj_matrix"]
	samples = data.shape[0]
	features = data.shape[1]
	values = data.data
	coo_data = data.tocoo()
	# indices = torch.LongTensor([coo_data.row, coo_data.col])
	indices = np.array([coo_data.row, coo_data.col])

	# Node matching
	N = 2708
	test_pairs = [[i, i] for i in range(N)]
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	node_features = loader["node_attr"]
	node_features = node_features.toarray()

	edge_index1 = torch.tensor(indices, dtype=torch.long)
	x1 = torch.tensor(node_features, dtype=torch.float)

	edge_index2 = torch.tensor(indices, dtype=torch.long)
	x2 = torch.tensor(node_features, dtype=torch.float)

	with_mean1 = False
	with_mean2 = True

elif raw_data_file == "flickr-lastfm":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-lastfm.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "douban":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/douban.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "flickr-myspace":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-myspace.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "ACM-DBLP":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/ACM-DBLP.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = True
	with_mean2 = True

elif raw_data_file == "cora":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/cora.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = False
	with_mean2 = True

# Create data
train_loader = {}

# Include edge_attr
subgroup_id = 1
data1 = Data(x=x1, edge_index=edge_index1)
if has_edge_attributes:
    data1.edge_attr = edge_attr1
else:
	data1 = add_edge_attributes(data1)
print(subgroup_id, data1)
info_data(data1, include_vectors=True)

subgroup_id = 2
data2 = Data(x=x2, edge_index=edge_index2)
if has_edge_attributes:
    data2.edge_attr = edge_attr2
else:
	data2 = add_edge_attributes(data2)
print(subgroup_id, data2)
info_data(data2, include_vectors=True)

# Normalize
normalize_node_edge_features(data1, data2, with_mean1=with_mean1, with_mean2=with_mean2) # Important

# Apply transform
subgroup_id = 1
data1 = transform(data1)
print(subgroup_id, data1)
# info(data1)

subgroup_id = 2
data2 = transform(data2)
print(subgroup_id, data2)
# info(data2)

# Include edge_weight
subgroup_id = 1
edge_index_norm1, edge_weight_norm1 = gcn_norm(
	data1.edge_index, num_nodes=data1.num_nodes, add_self_loops=False
)
data1.edge_index = edge_index_norm1
data1.edge_weight = edge_weight_norm1
train_loader[f"{group_id}_{subgroup_id}"] = data1
print(1, data1)

subgroup_id = 2
edge_index_norm2, edge_weight_norm2 = gcn_norm(
	data2.edge_index, num_nodes=data2.num_nodes, add_self_loops=False
)
data2.edge_index = edge_index_norm2
data2.edge_weight = edge_weight_norm2
train_loader[f"{group_id}_{subgroup_id}"] = data2
print(2, data2)

info_data(data1, include_vectors=True)
info_data(data2, include_vectors=True)

list_train_loader.append(train_loader)

Loading training datasets
1 Data(x=[9872, 17], edge_index=[2, 79122], edge_attr=[79122, 4])
Validate:	 True
Num. nodes:	 9872
Num. edges:	 79122
Num. features:	 17
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[   0,    0,    0,  ..., 9870, 9870, 9871],
        [   1,    2,    3,  ..., 1586, 2482, 4454]])
tensor([[61., 42., 16.,  ..., 14.,  1., 22.],
        [54., 18., 12.,  ..., 11.,  0., 16.],
        [54., 61.,  9.,  ..., 10.,  0., 39.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  1.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  1.]])
tensor([[5.0000e+00, 1.3661e-02, 1.9170e+00, 9.2901e-01],
        [3.2000e+01, 1.0667e-01, 1.0993e+01, 9.5597e-01],
        [1.4000e+01, 4.4304e-02, 4.1508e+00, 8.7719e-01],
        ...,
        [3.0000e+00, 2.1429e-01, 1.1592e+00, 8.1650e-01],
        [3.0000e+00, 2.7273e-01, 1.1148e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 4.0825e-01]])

In [14]:
df_node_alignment_truth

,0,1
0,9686,5356
1,1303,6620
2,3080,1745
3,5071,7441
4,2476,3459
...,...,...
6320,3770,7937
6321,6314,1944
6322,6379,136
6323,6689,2746


### Train

In [15]:
def fit_TGAE_subgraph_v1(encoder, dataset, no_samples, model, epochs, train_loader, lr, test_pairs=None):
	best_hitAtOne = 0
	best_hitAtFive = 0
	best_hitAtTen = 0
	best_hitAtFifty = 0
	list_loss = []

	optimizer = Adam(model.parameters(), lr=lr,weight_decay=5e-4)
	
	# Initialize early stopping
	patience = 50 # Change 
	delta = 1e-4 # 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		# Train
		model.train()
		loss = 0.0
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		optimizer.zero_grad()
		loss = loss / no_samples
		loss.backward()
		optimizer.step()

		loop_obj.set_postfix_str(f"Loss: {loss.item():.4f}")
		list_loss.append(loss.item())

		# Check early stopping condition
		early_stopping.check_early_stop(loss.item(), epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break

		# Evaluation (for firts dataset)
		""" model.eval()
		with torch.no_grad():
			keys = list(train_loader.keys())
			data1 = train_loader[keys[0]]
			data2 = train_loader[keys[1]]

			z1 = model(data1.x, data1.edge_index).detach()
			z2 = model(data2.x, data2.edge_index).detach()
			
			# Similarity matrix
			# option 1
			D = torch.cdist(z1, z2, 2)

			# option 2 (GPU problem)
			# D = 1 - F.cosine_similarity(z1.unsqueeze(1), z2.unsqueeze(0), dim=-1)

			# option 3 (Decoder cosine similarity)
			" "" z1n = F.normalize(z1, dim=1)
			z2n = F.normalize(z2, dim=1)
			D = 1 - (z1n @ z2n.T) " ""

			if dataset == "ACM_DBLP":
				test_idx = test_pairs[:, 0].astype(int)
				labels = test_pairs[:, 1].astype(int)
			else:
				test_idx = test_pairs[0, :].astype(int)
				labels = test_pairs[1, :].astype(int)
				
			hitAtOne = 0
			hitAtFive = 0
			hitAtTen = 0
			hitAtFifty = 0
			hitAtHundred = 0
			for i in range(len(test_idx)):
				dist_list = D[test_idx[i]]
				sorted_neighbors = torch.argsort(dist_list).cpu()
				label = labels[i]
				for j in range(100):
					if (sorted_neighbors[j].item() == label):
						if (j == 0):
							hitAtOne += 1
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 4):
							hitAtFive += 1
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 9):
							hitAtTen += 1
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 49):
							hitAtFifty += 1
							hitAtHundred += 1
							break
						elif (j <= 100):
							hitAtHundred += 1
							break
			cur_hitAtOne = hitAtOne / len(test_idx)
			cur_hitAtFive = hitAtFive / len(test_idx)
			cur_hitAtTen = hitAtTen / len(test_idx)
			cur_hitAtFifty = hitAtFifty / len(test_idx)

			if(cur_hitAtOne > best_hitAtOne): best_hitAtOne = cur_hitAtOne
			if (cur_hitAtFive > best_hitAtFive): best_hitAtFive = cur_hitAtFive
			if (cur_hitAtTen > best_hitAtTen): best_hitAtTen = cur_hitAtTen
			if (cur_hitAtFifty > best_hitAtFifty): best_hitAtFifty = cur_hitAtFifty

	print("The best results achieved:")
	print("Hit@1: ", end="")
	print(best_hitAtOne)
	print("Hit@5: ", end="")
	print(best_hitAtFive)
	print("Hit@10: ", end="")
	print(best_hitAtTen)
	print("Hit@50: ", end="")
	print(best_hitAtFifty) """

	# Evaluation (for others dataset)
	dict_node_embeddings = {}
	model.eval()
	with torch.no_grad():
		for ts in train_set:
			data = train_loader[ts]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			dict_node_embeddings[ts] = z.cpu().numpy()

	del loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()
	
	return dict_node_embeddings, list_loss

In [16]:
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set, no_samples, 
                    epochs, lr, weight_decay, patience):
	list_loss = []

	optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
	
	# Initialize early stopping
	# patience = 10
	delta = 1e-4
	warmup = 10
	early_stopping = EarlyStopping(patience=patience, delta=delta, warmup=warmup, verbose=True)

	# Best model checkpoint
	best_loss = float("inf")
	best_model_state = None

	""" model_dir = f"experiments/output/{exp}/models"
	os.makedirs(model_dir, exist_ok=True)

	best_model_path = (
		f"{model_dir}/best_{encoder}_{group_id}.pt"
	) """

	loop_obj = tqdm(range(1, epochs + 1))
	
	# Train mode
	model.train() # Set the model to training mode (enables dropout and BN updates)
	
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		
		optimizer.zero_grad() # Reset accumulated gradients before backpropagation
		
		loss = torch.tensor(0.0, device=device)
		
		for ts in random.sample(train_set, k=len(train_set)): # shuffle train_set
			data = train_loader[ts]

			# Encoder
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
				# z = F.normalize(z, dim=1)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			elif encoder == "GCNRes":
				z = model(data.x, data.edge_index, data.edge_weight)
			elif encoder == "GCN":
				z = model(data.x, data.edge_index, data.edge_weight)
			elif encoder == "GAT":
				z = model(data.x, data.edge_index)
			elif encoder == "GATE":
				z = model(data.x, data.edge_index, data.edge_attr)

			# Positive edges
			pos_edge_index = data.edge_index
			
			# Negative edges
			# option 1
			neg_edge_index = negative_sampling(
				edge_index=data.edge_index,
				num_nodes=z.size(0),
				num_neg_samples=pos_edge_index.size(1), # Change 2 to other value if needed
				method="sparse"
			)

			# option 2 Negative edges (dynamic)
			""" ratio = neg_ratio_schedule(epoch, epochs)
			num_neg = compute_num_neg_samples(
				edge_index=edge_index,
				num_nodes=z.size(0),
				ratio=ratio
			)
			neg_edge_index = negative_sampling(
				edge_index=edge_index,
				num_nodes=z.size(0),
				num_neg_samples=num_neg,
				method="sparse"
			) """
			
			# Decoder
			# option 1
			pos_logits = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
			neg_logits = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)
			
			# option 2
			""" pos_logits = F.cosine_similarity(
				z[pos_edge_index[0]],
				z[pos_edge_index[1]],
				dim=1
			)
			neg_logits = F.cosine_similarity(
				z[neg_edge_index[0]],
				z[neg_edge_index[1]],
				dim=1
			) """

			# Loss
			pos_labels = torch.ones_like(pos_logits)
			neg_labels = torch.zeros_like(neg_logits)

			# option 1
			""" loss_pos = binary_cross_entropy_with_logits(pos_logits, pos_labels)
			loss_neg = binary_cross_entropy_with_logits(neg_logits, neg_labels)
			loss += loss_pos + loss_neg """

			# option 2
			# num_pos = pos_edge_index.size(1)
			# num_neg = neg_edge_index.size(1)
			# pos_weight = torch.tensor([num_neg / num_pos], device=device)
			logits = torch.cat([pos_logits, neg_logits], dim=0)
			labels = torch.cat([pos_labels, neg_labels], dim=0)
			loss_temp = F.binary_cross_entropy_with_logits(logits, labels) #, pos_weight=pos_weight) # with pos_weight
			loss += loss_temp
			
		loss = loss / no_samples
		loss.backward() # Compute gradients of the loss w.r.t. model parameters
		optimizer.step() # Update model parameters using computed gradients

		current_loss = loss.item()

		loop_obj.set_postfix_str(f"Loss: {current_loss:.4f} | Best: {best_loss:.4f}")
	
		list_loss.append(current_loss)

		# Save the best model
		if current_loss < best_loss - delta:
			best_loss = current_loss
			best_model_state = copy.deepcopy(model.state_dict())
			""" best_model_state = {
				key: value.detach().cpu().clone()
				for key, value in model.state_dict().items()
			}
			torch.save(
				{
					"epoch": epoch,
					"best_loss": best_loss,
					"model_state_dict": best_model_state
				},
				best_model_path
			) """

		# Check early stopping condition
		early_stopping.check_early_stop(current_loss, epoch)
		if early_stopping.stop_training:
			print(f"Early stopping at epoch {epoch}")
			break
		# torch.cuda.empty_cache()

	# Restore the best model
	if best_model_state is not None:
		model.load_state_dict(best_model_state)
		print(f"Best epoch loss: {best_loss:.6f}")
		# print(f"Best model saved at: {best_model_path}")

	else:
		raise RuntimeError(
			"No valid model checkpoint was saved."
		)

	# Generate embeddings using the best model
	list_df_node_embeddings = []

	model.eval() # Set the model to evaluation mode (disables dropout and BN updates)
	with torch.no_grad():
		for subgroup_id in subgroups_id:

			# Get node embeddings
			data = train_loader[f"{group_id}_{subgroup_id}"]
			if encoder == "GIN":
				z = model(data.x, data.edge_index)
			elif encoder == "GINE":
				z = model(data.x, data.edge_index, data.edge_attr)
			elif encoder == "GCNRes":
				z = model(data.x, data.edge_index, data.edge_weight)
			elif encoder == "GCN":
				z = model(data.x, data.edge_index, data.edge_weight)
			elif encoder == "GAT":
				z = model(data.x, data.edge_index)
			elif encoder == "GATE":
				z = model(data.x, data.edge_index, data.edge_attr)
			
			df_node_embeddings = pd.DataFrame(z.cpu().numpy())
			df_node_embeddings["subgroup_id"] = [subgroup_id] * len(df_node_embeddings)
			
			# This is only for MS data (get node ids to replace original Id)
			""" df_nodes = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/nodes_{group_id}_{subgroup_id}.csv")
			# idx, id, mz, rt, intensity_mean, intensity_cv
			df_node_embeddings.insert(0, "Id", df_nodes["id"].values) """

			# This is only for no MS data (ids to replace with 0, 1, 2, ...)
			df_node_embeddings.insert(0, "Id", range(len(df_node_embeddings)))

			list_df_node_embeddings.append(df_node_embeddings)

		# Concat node embeddings
		df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)
		
		# Save node embeddings
		df_node_embeddings_concat.to_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", index=False)
	
	# Save loss
	np.save(f"experiments/output/{exp}/loss/{encoder}_{group_id}.npy", np.array(list_loss))

	del model, loss, z
	# torch.cuda.synchronize()
	torch.cuda.empty_cache()

In [17]:
encoders

['GCNRes', 'GIN', 'GINE', 'GCN', 'GAT', 'GATE']

In [18]:
list_train_set

[['ACM-DBLP_1', 'ACM-DBLP_2']]

In [19]:
list_train_loader

[{'ACM-DBLP_1': Data(x=[9872, 17], edge_index=[2, 79122], edge_attr=[79122, 4], edge_weight=[79122]),
  'ACM-DBLP_2': Data(x=[9916, 17], edge_index=[2, 89616], edge_attr=[89616, 4], edge_weight=[89616])}]

In [20]:
seed = 42
num_exp = 10 # Change

list_exp = []

for n_exp in range(num_exp):
	print(n_exp)
	#---
	# Train
	#---
	for encoder in encoders:
		for i, train_loader in enumerate(list_train_loader):

			group_id = groups_id[i]
			subgroup_id = subgroups_id[group_id]

			train_set = list_train_set[i]

			no_samples = len(train_set) # * (1 + 1)  # num datasets * num of samples by dataset 
			input_dim = train_loader[train_set[0]].num_node_features

			set_seed(seed + n_exp)
			if encoder == "GIN":
				model = TGAE_GIN(NUM_HIDDEN_LAYERS, 
							input_dim,
							HIDDEN_DIM,
							output_feature_size).to(device)
			elif encoder == "GINE":
				edge_dim = train_loader[train_set[0]].edge_attr.size(1)
				model = TGAE_GINE(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size, edge_dim).to(device)
			elif encoder == "GCNRes":
				model = TGAE_GCNRes(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size).to(device)
			elif encoder == "GCN":
				model = TGAE_GCN(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size).to(device)
			elif encoder == "GAT":
				model = TGAE_GATv2(NUM_HIDDEN_LAYERS, 
							input_dim,
							HIDDEN_DIM,
							output_feature_size).to(device)
			elif encoder == "GATE":
				edge_dim = train_loader[train_set[0]].edge_attr.size(1)
				model = TGAE_GATv2E(NUM_HIDDEN_LAYERS,
							input_dim,
							HIDDEN_DIM,
							output_feature_size, edge_dim).to(device)
			
			print("Fitting model")
			print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

			# Calculate elapsed time
			start_time = time.perf_counter()
			
			fit_TGAE_subgraph(model, encoder, dataset, group_id, subgroup_id, train_loader, train_set, no_samples, epochs, lr, weight_decay, patience)

			end_time = time.perf_counter()
			execution_time = end_time - start_time

			list_exp.append([raw_data_file, encoder, output_feature_size, NUM_HIDDEN_LAYERS, n_exp + 1, execution_time])

	#---
	# Similarity analysis
	#---
	K = 1 # Change
	for encoder in encoders:
		for group_id in groups_id:
			df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
			# Id, 0, 1,	2, ..., subgroup_id

			subgroups_id_ = subgroups_id[group_id]

			# Calculate distance matrix (KNN)
			knn = NearestNeighbors(n_neighbors=K, metric="euclidean")

			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
			x = df_node_embeddings.iloc[:, 1:-1].values

			df_node_alignment = pd.DataFrame()
			df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

			for subgroup_id in subgroups_id_[1:]:
				df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
				y = df_node_embeddings.iloc[:, 1:-1].values

				knn.fit(y)
				distances, indices = knn.kneighbors(x)
				indices = indices.squeeze() # (N,)

				df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
				
				""" df_node_alignment[f"distances"] = distances
				avg_distances = df_node_alignment["distances"].mean()
				df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
				df_node_alignment """
			
			# Save previus node alignment
			df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
			# display(df_node_alignment)

			# Find node alignment for all datasets
			""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
			list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
			# print(len(df_node_alignment_filter))
			# display(df_node_alignment_filter)
			print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
			
			# Save common node id
			# common_node_id = df_node_alignment_filter.iloc[:, 0].values
			# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

			# Save common node id as .csv
			df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
			df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
			df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

	# For cosine and L2
	for retrieval in ["cosine", "l2"]:
		for encoder in encoders:
			for group_id in groups_id:
				print(encoder, group_id, retrieval)
				df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
				# Id, 0, 1,	2, ..., subgroup_id

				df_node_embeddings1 = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == "1"]
				df_node_embeddings2 = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == "2"]

				z1 = df_node_embeddings1.iloc[:, 1:-1].values
				z2 = df_node_embeddings2.iloc[:, 1:-1].values

				sim = similarity_matrix(torch.tensor(z1), torch.tensor(z2), retrieval=retrieval, csls_k=None)
				match = hungarian_matching(sim, min_sim=None)

				list_1 = df_node_embeddings1.iloc[match[0]]["Id"].values
				list_2 = df_node_embeddings2.iloc[match[1]]["Id"].values

				df_node_alignment = pd.DataFrame([list_1, list_2])
				df_node_alignment= df_node_alignment.T

				df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]

				# Save common node id as .csv
				df_common_node_id = df_node_alignment_filter.iloc[:, [0]].copy()
				df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
				df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}_{retrieval}.csv", index=False, header=False)

	#---
	# Evaluation
	#---
	# list_accuracy = []
	# list_encoder = []
	
	for k, encoder in enumerate(encoders):
		for group_id in groups_id:
			# Read node alignment
			df_common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
			# df_common_node_id[[0, 1]] = np.sort(df_common_node_id[[0, 1]], axis=1)

			# Evaluation
			df_matching = df_common_node_id.merge(df_node_alignment_truth, on=[0, 1], how="inner")

			# Accuracy
			accuracy = len(df_matching) / len(df_node_alignment_truth)
			
			list_exp[len(encoders) * n_exp + k].append(accuracy)
			# list_accuracy.append(accuracy)
			# list_encoder.append(encoder)

	#---
	# Evaluation (based on PlanetAlign)
	#---

	# Get node embeddings
	""" dict_node_embeddings = {}
	for encoder in encoders:
		list_node_embeddings = []
		for group_id in groups_id:
			df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
			# Id, 0, 1,	2, ..., subgroup_id

			for subgroup_id in subgroups_id[group_id]:
				df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
				list_node_embeddings.append(df_node_embeddings)
		dict_node_embeddings[encoder] = list_node_embeddings """

	# Evaluation
	# list_metrics = []

	df_node_alignment_truth_ = torch.tensor(df_node_alignment_truth.values)

	""" for retrieval in ["cosine", "l2"]:
		for j, encoder in enumerate(encoders):
			list_node_embeddings = dict_node_embeddings[encoder]
			# Id, 0, 1,	2, ..., subgroup_id

			emb1 = list_node_embeddings[0].iloc[:, 1:-1]
			emb2 = list_node_embeddings[1].iloc[:, 1:-1]

			emb1_ = torch.tensor(emb1.values)
			emb2_ = torch.tensor(emb2.values)

			# Similarity matrix
			# Option 1
			# S = pairwise_cosine_similarity(emb1_, emb2_)

			# Option 2
			" "" S = cosine_similarity(emb1, emb2)
			S = torch.tensor(S) " ""

			# Option 3
			" "" S = pairwise_distances(emb1, emb2, metric="euclidean")
			S = torch.tensor(S)
			S = -S # convert distances matrix to similarity matrix "" "
			
			# Option 4
			" "" S = torch.cdist(emb1_, emb2_, 2)
			S = -S # convert distances matrix to similarity matrix " ""

			hits = hits_ks_scores(S, df_node_alignment_truth_, mode="mean")
			mrr = mrr_score(S, df_node_alignment_truth_, mode="mean")
			print(encoder, hits, mrr)

			list_exp[len(encoders) * n_exp + j] += list(hits.values()) + [mrr] """

	# For cosine and L2
	for retrieval in ["cosine", "l2"]:
		for k, encoder in enumerate(encoders):
			for group_id in groups_id:
				print(encoder, group_id, retrieval)
				df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
				# Id, 0, 1,	2, ..., subgroup_id

				df_node_embeddings1 = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == "1"]
				df_node_embeddings2 = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == "2"]

				z1 = df_node_embeddings1.iloc[:, 1:-1].values
				z2 = df_node_embeddings2.iloc[:, 1:-1].values

				S = similarity_matrix(torch.tensor(z1), torch.tensor(z2), retrieval=retrieval, csls_k=None)
    
				hits = hits_ks_scores(S, df_node_alignment_truth_, ks=KS, mode="mean")
				mrr = mrr_score(S, df_node_alignment_truth_, mode="mean")
				print(encoder, hits, mrr)
	
				list_exp[len(encoders) * n_exp + k] += list(hits.values()) + [mrr]

0
Fitting model
GCNRes exp106 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 92:  18%|█▊        | 91/500 [00:55<04:09,  1.64it/s, Loss: 0.4170 | Best: 0.4158]


Stopping early as no improvement has been observed.
Early stopping at epoch 92
Best epoch loss: 0.415849
Fitting model
GIN exp106 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 98:  19%|█▉        | 97/500 [02:32<10:34,  1.57s/it, Loss: 0.4057 | Best: 0.4051]


Stopping early as no improvement has been observed.
Early stopping at epoch 98
Best epoch loss: 0.405057
Fitting model
GINE exp106 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 216:  43%|████▎     | 215/500 [06:30<08:37,  1.82s/it, Loss: 0.4196 | Best: 0.4160]


Stopping early as no improvement has been observed.
Early stopping at epoch 216
Best epoch loss: 0.415974
Fitting model
GCN exp106 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 120:  24%|██▍       | 119/500 [01:26<04:36,  1.38it/s, Loss: 0.4235 | Best: 0.4152]


Stopping early as no improvement has been observed.
Early stopping at epoch 120
Best epoch loss: 0.415181
Fitting model
GAT exp106 ['ACM-DBLP_1', 'ACM-DBLP_2'] 0.0001 500 17 128 2


Epoch: 1:   0%|          | 0/500 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 390.00 MiB. GPU 1 has a total capacity of 47.41 GiB of which 128.81 MiB is free. Process 3798244 has 6.05 GiB memory in use. Process 2889493 has 11.55 GiB memory in use. Including non-PyTorch memory, this process has 29.40 GiB memory in use. Of the allocated memory 27.87 GiB is allocated by PyTorch, and 1.21 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Summary

In [ ]:
list_exp

[['ACM-DBLP',
  'GCNRes',
  128,
  12,
  1,
  48.95532459998503,
  0.566798418972332,
  0.5814229249954224,
  0.8071146309375763,
  0.869249016046524,
  0.9399209320545197,
  0.6792730391025543,
  0.5725691616535187,
  0.8008695542812347,
  0.8617391288280487,
  0.935494065284729,
  0.6711159348487854],
 ['ACM-DBLP',
  'GIN',
  128,
  12,
  1,
  122.8241990050301,
  0.5683794466403163,
  0.5720948576927185,
  0.7859288454055786,
  0.8432410955429077,
  0.9121739268302917,
  0.6641748249530792,
  0.576047420501709,
  0.7864031791687012,
  0.8427667915821075,
  0.9163636565208435,
  0.6666191518306732],
 ['ACM-DBLP',
  'GINE',
  128,
  12,
  1,
  226.48101623822004,
  0.5941501976284584,
  0.5923320055007935,
  0.8060078918933868,
  0.8665612637996674,
  0.9303557276725769,
  0.6842514276504517,
  0.5943873524665833,
  0.8081423044204712,
  0.8684585094451904,
  0.9306719303131104,
  0.6861178874969482],
 ['ACM-DBLP',
  'GCN',
  128,
  12,
  1,
  38.43493693508208,
  0.6039525691699604,


In [ ]:
["Dataset", "Encoder", "Runtime", "Accuracy"] + [f"Hits@{k}-cosine" for k in KS] +  ["MRR"] + [f"Hits@{k}-L2" for k in KS] +  ["MRR"]

['Dataset',
 'Encoder',
 'Runtime',
 'Accuracy',
 'Hits@1-cosine',
 'Hits@5-cosine',
 'Hits@10-cosine',
 'Hits@50-cosine',
 'MRR',
 'Hits@1-L2',
 'Hits@5-L2',
 'Hits@10-L2',
 'Hits@50-L2',
 'MRR']

In [ ]:
columns = ["Dataset", "Encoder", "Dimension", "Layer", "Seed", "Runtime", "Accuracy"] + [f"Hits@{k}-cosine" for k in list(hits.keys())] +  ["MRR-cosine"] + [f"Hits@{k}-L2" for k in list(hits.keys())] +  ["MRR-L2"]
df_summary = pd.DataFrame(list_exp, columns=columns)
df_summary.sort_values(by=["Encoder", "Seed"], inplace=True)
df_summary

,Dataset,Encoder,Dimension,Layer,Seed,Runtime,Accuracy,Hits@1-cosine,Hits@5-cosine,Hits@10-cosine,Hits@50-cosine,MRR-cosine,Hits@1-L2,Hits@5-L2,Hits@10-L2,Hits@50-L2,MRR-L2
4,ACM-DBLP,GAT,128,12,1,238.878117,0.514150,0.525375,0.758340,0.824664,0.912964,0.626372,0.521028,0.752411,0.818024,0.903794,0.621157
10,ACM-DBLP,GAT,128,12,2,255.636857,0.498814,0.511858,0.742213,0.812174,0.904111,0.612673,0.507668,0.736522,0.805534,0.893597,0.607353
5,ACM-DBLP,GATE,128,12,1,180.118124,0.477945,0.483874,0.714783,0.782688,0.884664,0.585243,0.482292,0.705613,0.776127,0.877628,0.580753
11,ACM-DBLP,GATE,128,12,2,184.212332,0.492174,0.508933,0.739368,0.806877,0.899526,0.608624,0.503557,0.731225,0.798814,0.889723,0.602235
3,ACM-DBLP,GCN,128,12,1,38.434937,0.603953,0.600553,0.824190,0.879842,0.948538,0.696531,0.606008,0.824348,0.881818,0.950514,0.699903
9,ACM-DBLP,GCN,128,12,2,37.446549,0.599526,0.598340,0.822451,0.881660,0.947431,0.695010,0.599526,0.823715,0.882767,0.948696,0.696251
0,ACM-DBLP,GCNRes,128,12,1,48.955325,0.566798,0.581423,0.807115,0.869249,0.939921,0.679273,0.572569,0.800870,0.861739,0.935494,0.671116
6,ACM-DBLP,GCNRes,128,12,2,37.286597,0.580553,0.594466,0.815257,0.873676,0.941818,0.690084,0.584822,0.806245,0.866245,0.937549,0.680837
1,ACM-DBLP,GIN,128,12,1,122.824199,0.568379,0.572095,0.785929,0.843241,0.912174,0.664175,0.576047,0.786403,0.842767,0.916364,0.666619
7,ACM-DBLP,GIN,128,12,2,127.321672,0.569644,0.571462,0.782530,0.840079,0.910277,0.662568,0.574308,0.782372,0.838814,0.912016,0.664414


In [ ]:
df_text = df_summary.map(lambda x: f"{x}".replace(".", ","))
df_text

In [ ]:
print(NUM_HIDDEN_LAYERS)
print(HIDDEN_DIM)
print(output_feature_size)